In [6]:
!pip install -q kaggle

In [8]:
!kaggle datasets download -d grassknoted/asl-alphabet -p /content/asl --unzip

Dataset URL: https://www.kaggle.com/datasets/grassknoted/asl-alphabet
License(s): GPL-2.0
100% 1.03G/1.03G [00:08<00:00, 125MB/s] 



In [9]:
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split
import torch

raw = Path("/content/asl/asl_alphabet_train/asl_alphabet_train")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = datasets.ImageFolder(raw, transform=transform)

# Select 200 images per class
indices = []
for class_index in range(len(dataset.classes)):
    class_indices = [
        i for i, (_, label) in enumerate(dataset.samples)
        if label == class_index
    ]
    indices.extend(class_indices[:200])

small_dataset = Subset(dataset, indices)

train_size = int(0.8 * len(small_dataset))
val_size = len(small_dataset) - train_size

train_dataset, val_dataset = random_split(
    small_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

num_classes = len(dataset.classes)

print("Number of classes:", num_classes)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

Number of classes: 29
Training images: 4640
Validation images: 1160


Libraries

In [10]:
import torch
import torch.nn as nn
from torchvision import models
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 29

Load pretrained AlexNet Model

In [11]:
model = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)

# Freeze pretrained feature layers
for param in model.features.parameters():
    param.requires_grad = False

# Replace the final classification layer
model.classifier[6] = nn.Linear(4096, num_classes)

model = model.to(device)

print("Device:", device)
print("Classes:", num_classes)

Device: cpu
Classes: 29


Loss and optimizer

In [12]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=0.0001
)

Train AlexNet

In [13]:
epochs = 3
start_time = time.time()

for epoch in range(epochs):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # Make predictions
        outputs = model(images)

        # Calculate prediction error
        loss = criterion(outputs, labels)

        # Update the model
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Calculate accuracy and loss
        total_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = total_loss / total
    epoch_accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch + 1}/{epochs}] "
        f"Loss: {epoch_loss:.4f} "
        f"Accuracy: {epoch_accuracy:.2f}%"
    )

training_time = time.time() - start_time
print(f"Training time: {training_time:.2f} seconds")

Epoch [1/3] Loss: 0.7138 Accuracy: 81.03%
Epoch [2/3] Loss: 0.0874 Accuracy: 97.39%
Epoch [3/3] Loss: 0.0556 Accuracy: 98.21%
Training time: 1305.95 seconds


Evaluate AlexNet

In [14]:
model.eval()

val_loss = 0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

val_loss /= total
val_accuracy = 100 * correct / total

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.2f}%")

Validation Loss: 0.0778
Validation Accuracy: 97.41%


Saving AlexNet

In [18]:
from pathlib import Path

Path("/content/results").mkdir(parents=True, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "classes": dataset.classes
}, "/content/results/alexnet_asl.pth")

print("ResNet18 model saved!")

ResNet18 model saved!


In [20]:
from google.colab import files

files.download("/content/results/alexnet_asl.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>